# Notebook 01 — Data Ingestion

**Goal**: Download all data sources and load them into PostgreSQL.

**Sources**:
- NYC TLC Yellow Taxi trip records (Parquet)
- Open-Meteo historical weather (free API)
- NYC events calendar (curated CSV)
- FRED economic indicators (API)

**After this notebook**: the `taxi_trips`, `weather_hourly`, `events`, and
`economic_indicators` tables are populated, and the `ml_features` materialized
view has been refreshed and is ready for EDA.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from sqlalchemy import text
from src.utils.db import get_engine, refresh_materialized_view
from src.utils.logger import get_logger

logger = get_logger('notebook-01')
engine = get_engine()
print('Connected to database.')

## 1. Verify database schema

In [ ]:
# Check all tables exist
with engine.connect() as conn:
    tables = pd.read_sql(
        text("SELECT tablename FROM pg_tables WHERE schemaname = 'public'"),
        conn
    )
print('Tables in database:')
print(tables['tablename'].tolist())

## 2. Ingest NYC Taxi data (6 months)

In [ ]:
from src.ingest.taxi import ingest_months

months = ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06']
ingest_months(months)

# Verify row count
with engine.connect() as conn:
    count = pd.read_sql(text('SELECT COUNT(*) AS n FROM taxi_trips'), conn)
print(f'Taxi rows loaded: {count["n"].iloc[0]:,}')

## 3. Ingest weather data

In [ ]:
from src.ingest.weather import ingest_weather

ingest_weather('2024-01-01', '2024-06-30')

with engine.connect() as conn:
    count = pd.read_sql(text('SELECT COUNT(*) AS n FROM weather_hourly'), conn)
print(f'Weather rows loaded: {count["n"].iloc[0]:,}')

## 4. Ingest events

In [ ]:
from src.ingest.events import ingest_events

ingest_events()

with engine.connect() as conn:
    events = pd.read_sql(text('SELECT * FROM events ORDER BY event_date'), conn)
print(f'Events loaded: {len(events)}')
events.head(10)

## 5. Ingest economic indicators

In [ ]:
from src.ingest.economic import ingest_economic

ingest_economic('2024-01-01', '2024-06-30')

with engine.connect() as conn:
    econ = pd.read_sql(text('SELECT * FROM economic_indicators ORDER BY month_start'), conn)
econ

## 6. Refresh materialized view

In [ ]:
refresh_materialized_view('ml_features')

with engine.connect() as conn:
    count = pd.read_sql(text('SELECT COUNT(*) AS n FROM ml_features'), conn)
print(f'ML feature rows: {count["n"].iloc[0]:,}')
print('Ready for EDA in notebook 02.')